In [1]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg,tool
from typing import Annotated
from dotenv import load_dotenv
import os
import requests

/home/kamadahanam/Documents/ALL_WORKS/Programming/LangChain/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
load_dotenv('../.env')
EXCHANGE_RATE_KEY = os.getenv('EXCHANGE_RATE_KEY')

In [3]:
@tool
def get_conversion_factor(base_currency : str, target_currency: str)-> float:
    """
    This function returns the conversion factor between base_currency and target_currency.
    """
    url = f'https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_KEY}/pair/{base_currency}/{target_currency}'
    response = requests.get(url)

    return response.json()

In [4]:
get_conversion_factor.args

{'base_currency': {'title': 'Base Currency', 'type': 'string'},
 'target_currency': {'title': 'Target Currency', 'type': 'string'}}

In [5]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1769385601,
 'time_last_update_utc': 'Mon, 26 Jan 2026 00:00:01 +0000',
 'time_next_update_unix': 1769472001,
 'time_next_update_utc': 'Tue, 27 Jan 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 91.8161}

In [6]:
@tool
# Annotated[float, InjectedToolArg], this says, LLm will not fill this argument, I(developer/Runtime) will fill this argument
# If this not used, llm will just do both tools parallel, won't get conversion factor on time, and fill with it's parametric knowledge about the conversion factor
def convert(base_currency_value: float, conversion_rate: Annotated[float, InjectedToolArg])-> float:
    """
    This function converts the target_currency value of base_currency value ny returning the product of conversion rate and base_currency value
    """
    return base_currency_value * conversion_rate

In [7]:
# creating LLM
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm= HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3.1",
    task="text-generation",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
)

model = ChatHuggingFace(llm=llm)

In [8]:
llm_with_tools = model.bind_tools([get_conversion_factor,convert])

In [9]:
messages=[]
messages.append(HumanMessage('What is the Conversion Factor between INR and IRR. Convert 1 Lac INR into IRR'))

In [10]:
response=llm_with_tools.invoke(messages)
response.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'IRR'},
  'id': 'bd02fa8c503546fcaf59504a2cabe484',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 100000},
  'id': '5ad6017990114ff28c9360845c1bfa88',
  'type': 'tool_call'}]

In [11]:
messages.append(response)

In [17]:
import json

for call in response.tool_calls:
    if call['name'] =='get_conversion_factor':
        result1=get_conversion_factor.invoke(call)
        conversion_rate= json.loads(result1.content)['conversion_rate']
        messages.append(result1)
    if call['name'] =='convert':
        call['args']['conversion_rate'] = conversion_rate
        result2=convert.invoke(call)
        messages.append(result2)

In [18]:
messages

[HumanMessage(content='What is the Conversion Factor between INR and IRR. Convert 1 Lac INR into IRR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "INR", "target_currency": "IRR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'bd02fa8c503546fcaf59504a2cabe484', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 100000}', 'name': 'convert', 'description': None}, 'id': '5ad6017990114ff28c9360845c1bfa88', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 633, 'total_tokens': 712}, 'model_name': 'deepseek-ai/DeepSeek-V3.1', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bfa4a-c2a0-7562-beab-96012df45e7b-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'IRR'}, 'id': 'bd02fa8c503546fcaf59504a2cabe484', 'typ

In [19]:
llm_with_tools.invoke(messages).content

'The conversion factor between Indian Rupee (INR) and Iranian Rial (IRR) is 1 INR = 456.6952 IRR.\n\nConverting 1 Lac INR (which is 100,000 INR) to IRR:\n100,000 INR × 456.6952 = 45,669,520 IRR\n\nSo, 1 Lac INR is equivalent to approximately 45,669,520 Iranian Rial.'